# 04 Root Finding

[![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](LICENSE) [![Content License: CC BY 4.0](https://img.shields.io/badge/Content%20License-CC%20BY%204.0-blue.svg)](https://creativecommons.org/licenses/by/4.0/)

[![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](../LICENSE)
[![Content License: CC BY 4.0](https://img.shields.io/badge/Content%20License-CC%20BY%204.0-blue.svg)](https://creativecommons.org/licenses/by/4.0/)

In [ ]:
# === Environment Setup ===
import os
import sys
import math
import time
import random
import json
import textwrap
import warnings
from typing import Callable
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import brentq, newton, root

# --- Configuration ---
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 12, 'axes.titlesize': 'x-large',
    'axes.labelsize': 'large', 'xtick.labelsize': 'medium', 'ytick.labelsize': 'medium'})
np.set_printoptions(suppress=True, linewidth=120, precision=8)
warnings.filterwarnings('ignore', category=FutureWarning)
print("Environment initialized.")

## Part 2: Core Numerical Methods
## Chapter 2.4: Root-Finding and Fixed-Point Algorithms

### Table of Contents
1.  [Fixed-Point Theory: Existence and Uniqueness](#1.-Fixed-Point-Theory:-Existence-and-Uniqueness)
    *   [1.1 Fixed-Point Iteration](#1.1-Fixed-Point-Iteration)
    *   [1.2 Contraction Mapping Theorem](#1.2-Contraction-Mapping-Theorem)
    *   [1.3 Brouwer's Fixed-Point Theorem](#1.3-Brouwer's-Fixed-Point-Theorem)
    *   [1.4 Accelerating Convergence: Aitken's Delta-Squared Process](#1.4-Accelerating-Convergence:-Aitken's-Delta-Squared-Process)
2.  [Root-Finding Algorithms for Single Equations](#2.-Root-Finding-Algorithms-for-Single-Equations)
    *   [2.1 Bracketing vs. Open Methods](#2.1-Bracketing-vs.-Open-Methods)
    *   [2.2 Newton's Method: Basins of Attraction](#2.2-Newton's-Method:-Basins-of-Attraction)
3.  [Systems of Non-Linear Equations](#3.-Systems-of-Non-Linear-Equations)
    *   [3.1 Homotopy Continuation Methods](#3.1-Homotopy-Continuation-Methods)
    *   [3.2 Application: General Equilibrium with CES Utility](#3.2-Application:-General-Equilibrium-with-CES-Utility)
4.  [Summary](#4.-Summary)
5.  [Exercises](#5.-Exercises)

# The Lens: Equilibrium is a Zero

**What problem are we solving?**
In economics, equilibrium is everything. A market clears when supply equals demand ($S(p) - D(p) = 0$). A firm maximizes profit where marginal revenue equals marginal cost ($MR(q) - MC(q) = 0$). A macroeconomy is in steady state when capital accumulation ceases ($s f(k) - \delta k = 0$). All of these are **root-finding problems**. We are looking for the value of a variable $x$ that makes a function $f(x)$ equal to zero.

**Why this method?**
Analytical solutions are rare. For most realistic economic models, we must find the root numerically.
*   **Bisection:** Guaranteed convergence but slow.
*   **Newton-Raphson:** Quadratic convergence (fast) but can be unstable.
*   **Brent's Method:** The industrial standard that combines the best of both.

This notebook gives you the tools to solve the fundamental equation of economics: $f(x) = 0$.

### 1. Fixed-Point Theory: Existence and Uniqueness

#### 1.1 Fixed-Point Iteration
A **fixed point** of a function $g: X \to X$ is a point $x^* \in X$ such that $g(x^*) = x^*$. Many dynamic economic models are expressed as a law of motion $x_{t+1} = g(x_t)$, and the steady state of this system is a fixed point of $g$.

The simplest way to find a fixed point is via **fixed-point iteration**: choose an initial guess $x_0$ and then iterate $x_{k+1} = g(x_k)$ until the sequence converges.

#### 1.2 Contraction Mapping Theorem
**The Contraction Mapping Theorem** provides *sufficient* conditions for the existence and uniqueness of a fixed point and for the convergence of this iterative process. A function $g$ is a **contraction mapping** on a complete metric space $(X, d)$ if there exists a constant $\beta \in [0, 1)$ such that for all $x, y \in X$:
$$ d(g(x), g(y)) \le \beta \cdot d(x, y) $$
If $g$ is a contraction, then it has a unique fixed point, and fixed-point iteration will converge to it from any starting point. This is the workhorse theorem for proving convergence in dynamic programming.

#### 1.3 Brouwer's Fixed-Point Theorem
\nThis theorem was famously developed by L.E.J. Brouwer in the early 20th century. One of its most intuitive interpretations is the 'stirring coffee' analogy: no matter how you stir a cup of coffee (continuously, without tearing the liquid), some point of the liquid will end up in exactly the same position it started in.\nBrouwer's theorem provides a weaker but more general condition for the *existence* of a fixed point. It states that if $S$ is a **compact** (closed and bounded) and **convex** subset of a Euclidean space and $g: S \to S$ is a **continuous** function, then $g$ must have at least one fixed point.

**Key Differences from Contraction Mapping:**
- **Existence Only:** Brouwer's theorem does not guarantee uniqueness, nor does it tell us how to find the fixed point. It is non-constructive.
- **Weaker Conditions:** It does not require a metric or the contraction property, only continuity and a mapping from a compact, convex set to itself. This makes it applicable to a wider range of problems, such as proving the existence of a Nash Equilibrium in game theory.

#### 1.4 Accelerating Convergence: Aitken's Delta-Squared Process
Standard fixed-point iteration can be slow if the convergence is only linear. **Aitken's Delta-Squared Process** is a method for accelerating the convergence of such sequences. Given a sequence $\{x_k\}$, it generates a new, faster-converging sequence $\{x'_k\}$ using the formula:
$$ x'_k = x_k - \frac{(x_{k+1} - x_k)^2}{x_{k+2} - 2x_{k+1} + x_k} $$
This process can often turn a linearly convergent sequence into one that converges quadratically.\nIntuitively, the process uses three consecutive points ($x_k, x_{k+1}, x_{k+2}$) to estimate the rate of convergence and then extrapolates to the limit of the sequence. It's closely related to the Secant method for root-finding.

![Accelerating Solow Model Convergence with Aitken's Method](../images/02-Numerical-Methods/aitken_acceleration.png)

### 2. Root-Finding Algorithms for Single Equations

#### 2.1 Bracketing vs. Open Methods
Root-finding algorithms fall into two main categories:
- **Bracketing Methods (e.g., Bisection):** These require an initial interval $[a, b]$ where the root is known to exist (i.e., $f(a)$ and $f(b)$ have opposite signs). They are guaranteed to converge (**global convergence**) but are often slow (Bisection is linear).
- **Open Methods (e.g., Newton's, Secant):** These require only an initial guess. They use local information (derivatives) to find the next iterate. They are much faster (Newton's is quadratic) but are only guaranteed to converge if the initial guess is sufficiently close to the root (**local convergence**).

### Application: Finding a Bond's Yield to Maturity
A classic application of root-finding in finance is calculating a bond's **Yield to Maturity (YTM)**. The price of a bond is the present value of its future cash flows, which consist of periodic coupon payments and the final face value payment. The YTM is the single discount rate (the root) that equates the present value of these cash flows to the bond's current market price.

The price $P$ of a bond with face value $F$, coupon rate $c$, $T$ periods to maturity, and yield $y$ is:
$$ P(y) = \sum_{t=1}^{T} \frac{c F}{(1+y)^t} + \frac{F}{(1+y)^T} $$
Given a market price $P_{market}$, the YTM is the value of $y$ that solves the equation $P(y) - P_{market} = 0$.

In [ ]:
### Calculating Yield to Maturity

def bond_price(y, F=1000, c=0.05, T=10):
    """Calculates the price of a bond for a given yield."""
    coupon_pv = sum(c * F / (1 + y)**t for t in range(1, T + 1))
    face_value_pv = F / (1 + y)**T
    return coupon_pv + face_value_pv

market_price = 950.0

# The objective function is the difference between the theoretical price and the market price.
objective = lambda y: bond_price(y) - market_price

# Use a robust bracketing method to find the root.
ytm = brentq(objective, a=0.0, b=0.2)

print(f"> **Note:** For a market price of ${market_price:.2f}, the calculated Yield to Maturity is {ytm:.4%}.")

# --- Visualization ---
yields = np.linspace(0.01, 0.1, 100)
prices = [bond_price(y) for y in yields]

plt.figure(figsize=(10, 6))
plt.plot(yields, prices, label='Bond Price-Yield Curve')
plt.axhline(market_price, color='red', linestyle='--', label=f'Market Price (${market_price:.2f})')
plt.axvline(ytm, color='green', linestyle='--', label=f'YTM ({ytm:.2%})')
plt.title('Bond Price vs. Yield to Maturity')
plt.xlabel('Yield to Maturity (YTM)')
plt.ylabel('Price ($)')
plt.legend()
plt.grid(True)
plt.show()

#### 2.2 Newton's Method: Basins of Attraction
The sensitivity of Newton's method to the initial guess can be visualized by plotting its **basins of attraction**. For a function with multiple roots, the basin of attraction for a given root is the set of all starting points from which the method converges to that root. These basins can have intricate, fractal boundaries.

A classic example is finding the roots of $f(z) = z^3 - 1$ in the complex plane. The three roots are $1$, $e^{i2\pi/3}$, and $e^{-i2\pi/3}$. The plot below shows which root Newton's method converges to for each starting point in the complex plane.

![Basins of Attraction for Newton's Method](../images/02-Numerical-Methods/newton_basins.png)

### 3. Systems of Non-Linear Equations

#### 3.1 Homotopy Continuation Methods
Finding the root of a system of non-linear equations, $F(\mathbf{x}) = \mathbf{0}$, can be very difficult. Standard methods like multivariate Newton's method are highly sensitive to the initial guess. **Homotopy continuation** provides a more robust, global approach.

The idea is to start with a simple system $G(\mathbf{x})=\mathbf{0}$ that we *can* solve (e.g., $G(\mathbf{x}) = \mathbf{x} - \mathbf{x}_0$). We then define a **homotopy**, a function that gradually deforms $G$ into our target function $F$:
$$ H(\mathbf{x}, t) = (1-t)G(\mathbf{x}) + tF(\mathbf{x}), \quad t \in [0, 1] $$

We start with the known solution to $H(\mathbf{x}, 0) = G(\mathbf{x}) = \mathbf{0}$. We then increment $t$ in small steps, and at each step, we use the solution from the previous step as the initial guess to solve $H(\mathbf{x}, t_{new}) = \mathbf{0}$. By tracing this path of solutions from $t=0$ to $t=1$, we arrive at the solution to the original problem, $H(\mathbf{x}, 1) = F(\mathbf{x}) = \mathbf{0}$.

#### 3.2 Application: General Equilibrium with CES Utility
We can use `scipy.optimize.root` to solve for the equilibrium prices in a more complex general equilibrium model. Consider an Edgeworth box economy with two agents (A, B) and two goods (1, 2). Instead of Cobb-Douglas utility, we use the more general **Constant Elasticity of Substitution (CES)** utility function:
$$ U(x_1, x_2) = (\alpha x_1^\rho + (1-\alpha)x_2^\rho)^{1/\rho} $$
The elasticity of substitution is $\sigma = 1/(1-\rho)$. The demand functions derived from CES utility are more non-linear than those from Cobb-Douglas. We will normalize the price of good 2 to be the numéraire ($p_2=1$) and solve for the relative price $p_1$ that clears the market for good 1.

In [ ]:
### Solving for General Equilibrium with CES Utility

# Agent A: high preference for good 1, low elasticity
alpha_A, rho_A = 0.7, -1.0 # sigma = 0.5
# Agent B: balanced preference, high elasticity
alpha_B, rho_B = 0.5, 0.8 # sigma = 5.0

# Endowments [good1, good2]
e_A, e_B = np.array([4, 1]), np.array([1, 4])

def ces_demand(p1, p2, income, alpha, rho):
    """Marshallian demand for good 1 from CES utility."""
    sigma = 1 / (1 - rho)
    term1 = alpha**sigma / (alpha**sigma * p1**sigma + (1-alpha)**sigma * p2**sigma)
    return term1 * income / p1

def excess_demand_good1(p1, p2=1.0):
    income_A = p1 * e_A[0] + p2 * e_A[1]
    income_B = p1 * e_B[0] + p2 * e_B[1]
    demand_A1 = ces_demand(p1, p2, income_A, alpha_A, rho_A)
    demand_B1 = ces_demand(p1, p2, income_B, alpha_B, rho_B)
    return (demand_A1 + demand_B1) - (e_A[0] + e_B[0])

print("> **Note:** Solving for the root of the more complex excess demand function:")
p1_star = brentq(excess_demand_good1, a=0.1, b=10.0)
p2_star = 1.0

print(f"> **Note:** Equilibrium found! Relative price p1/p2: {p1_star/p2_star:.4f}")
print(f"> **Note:** Excess demand for good 1 at this price: {excess_demand_good1(p1_star):.2e}")

# Summary

Root finding is the computational engine of equilibrium analysis.

**Key Takeaways:**
*   **Use Brent's Method:** For scalar problems, `scipy.optimize.brentq` is the gold standard. It is robust and fast.
*   **Provide Bounds:** Algorithms work best when you can bracket the solution (e.g., knowing the price must be between 0 and 100).
*   **Newton requires Derivatives:** If you have the derivative, Newton's method is lightning fast. If not, use Secant or Brent.

## 5. Exercises

1.  **Aitken's Method from Scratch:** The code for `aitken_accelerate` is provided. Explain in words what each part of the formula for $x'_k$ is doing. Why might the denominator become close to zero, and what does this imply about the underlying sequence?

2.  **Implement Newton's Method for Systems:** Write a Python function that implements Newton's method for a system of non-linear equations. It should take a function `F` that returns a vector of errors, a function `J` that returns the Jacobian matrix, and an initial guess `x0`. Test it by solving the CES general equilibrium problem from Section 3.2.

3.  **Newton's Method Failure:** The function $f(x) = x^3 - 2x + 2$ has a local minimum near $x \approx 0.8$. Show that if you start Newton's method with an initial guess of $x_0=1$, the next iterate is $x_1=0$, where the derivative is zero, causing the method to fail. If you start at $x_0=0$, what happens?

4.  **Implicit Yield Curve:** The price `P` of a bond with face value `F`, coupon rate `c`, `T` periods to maturity, and yield-to-maturity `y` is given by: $ P = \sum_{t=1}^{T} \frac{c F}{(1+y)^t} + \frac{F}{(1+y)^T} $. Finding the yield-to-maturity (YTM) for a given market price `P` is a root-finding problem. Write a function that takes `P`, `F`, `c`, and `T` and uses `brentq` to solve for the YTM, `y`.

5.  **Homotopy Continuation:** Consider the difficult root-finding problem $F(x) = \cos(4\pi x) - x = 0$. It has many roots. Let your simple, solvable problem be $G(x) = x - 0.5 = 0$. Implement a simple homotopy continuation method to track the solution from $t=0$ to $t=1$. At each step of $t$, use `scipy.optimize.newton` to solve for the root of $H(x,t)$, using the previous step's solution as the initial guess. Plot the path of the root as a function of $t$.